# 🪙 Crypto Price Snapshot
## Lalit Gupta Batch T-361

**Objective:** Fetch live Top 50 cryptocurrency data from the CoinGecko public API and perform basic data analysis.

**Libraries Used:** `requests`, `json`, `csv`, `datetime` (no pandas needed!)

**API Used:** CoinGecko — Free, No API Key Required ✅

---

## 1. Install and Import Required Libraries

In [37]:
!pip install requests


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [38]:
import requests
import json
import csv
import os
import webbrowser
from datetime import datetime

print('✅ All libraries imported successfully!')
print(f"📅 Project run date: {datetime.now().strftime('%d-%m-%Y %H:%M:%S')}")

✅ All libraries imported successfully!
📅 Project run date: 23-04-2026 08:41:01


## 2. Fetch Data from CoinGecko API

In [39]:
API_URL = 'https://api.coingecko.com/api/v3/coins/mark'
PARAMS = {
    'vs_currency': 'usd',
    'order': 'market_cap_desc',
    'per_page': 50,
    'page': 1,
    'sparkline': False,
    'price_change_percentage': '24h'
}
HEADERS = {'User-Agent': 'Mozilla/5.0', 'Accept': 'application/json'}

def fetch_crypto_data():
    try:
        print('⏳ Fetching data from CoinGecko API...')
        response = requests.get(API_URL, params=PARAMS, headers=HEADERS, timeout=15)
        response.raise_for_status()
        data = response.json()
        print(f'✅ Successfully fetched {len(data)} coins!')
        return data
    except Exception as e:
        print(f'❌ Error: {e}')
        return None

raw_data = fetch_crypto_data()

⏳ Fetching data from CoinGecko API...
❌ Error: 404 Client Error: Not Found for url: https://api.coingecko.com/api/v3/coins/mark?vs_currency=usd&order=market_cap_desc&per_page=50&page=1&sparkline=False&price_change_percentage=24h


## 3. Parse and Clean Data

In [40]:
def safe(val, default=0):
    """Return val if not None, else default — fixes NoneType errors"""
    return val if val is not None else default

def parse_crypto_data(raw_data):
    coins = []
    for coin in raw_data:
        parsed = {
            'rank':               safe(coin.get('market_cap_rank'), 0),
            'name':               safe(coin.get('name'), 'N/A'),
            'symbol':             safe(coin.get('symbol'), 'N/A').upper(),
            'image':              safe(coin.get('image'), ''),
            'current_price_usd':  safe(coin.get('current_price'), 0),
            'market_cap_usd':     safe(coin.get('market_cap'), 0),
            'total_volume_usd':   safe(coin.get('total_volume'), 0),
            'high_24h_usd':       safe(coin.get('high_24h'), 0),
            'low_24h_usd':        safe(coin.get('low_24h'), 0),
            'price_change_24h':   round(safe(coin.get('price_change_percentage_24h'), 0), 2),
            'circulating_supply': safe(coin.get('circulating_supply'), 0),
            'ath_usd':            safe(coin.get('ath'), 0),
            'fetched_at':         datetime.now().strftime('%d-%m-%Y %H:%M:%S')
        }
        coins.append(parsed)
    return coins

if raw_data:
    coins = parse_crypto_data(raw_data)
    print(f'✅ Parsed {len(coins)} coins successfully!')
    print(f"Sample — {coins[0]['name']}: ${coins[0]['current_price_usd']:,.2f} | 24h: {coins[0]['price_change_24h']:+.2f}%")

## 4. Save to CSV

In [41]:
def save_to_csv(coins, filename='crypto_data.csv'):
    headers = ['rank','name','symbol','current_price_usd','market_cap_usd',
               'total_volume_usd','high_24h_usd','low_24h_usd',
               'price_change_24h','circulating_supply','ath_usd','fetched_at']
    with open(filename, 'w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=headers, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(coins)
    print(f"✅ Data saved to '{filename}'")

    save_to_csv(coins)

## 5. Data Analysis

In [42]:
prices        = [c['current_price_usd'] for c in coins]
changes       = [c['price_change_24h']  for c in coins]
gainers       = sorted(coins, key=lambda x: x['price_change_24h'], reverse=True)
losers        = sorted(coins, key=lambda x: x['price_change_24h'])
top_mc        = sorted(coins, key=lambda x: x['market_cap_usd'], reverse=True)
top_vol       = sorted(coins, key=lambda x: x['total_volume_usd'], reverse=True)
gainers_count = sum(1 for c in coins if c['price_change_24h'] > 0)
losers_count  = sum(1 for c in coins if c['price_change_24h'] < 0)
green_pct     = (gainers_count / len(coins)) * 100
avg_price     = sum(prices) / len(prices)
avg_change    = sum(changes) / len(changes)

if green_pct >= 70:   mood = 'VERY BULLISH 🚀'
elif green_pct >= 55: mood = 'BULLISH 📈'
elif green_pct >= 45: mood = 'NEUTRAL 😐'
elif green_pct >= 30: mood = 'BEARISH 📉'
else:                 mood = 'VERY BEARISH 💥'

print('📊 ANALYSIS COMPLETE')
print(f"  Top Gainer  : {gainers[0]['name']} (+{gainers[0]['price_change_24h']:.2f}%)")
print(f"  Top Loser   : {losers[0]['name']} ({losers[0]['price_change_24h']:.2f}%)")
print(f'  Market Mood : {mood}')
print(f'  Avg Price   : ${avg_price:,.2f}')
print(f'  Avg Change  : {avg_change:+.2f}%')

NameError: name 'coins' is not defined

## 6. 🌐 Generate HTML Dashboard & Open in Browser

In [ ]:
def fmt_price(p):
    p = p or 0
    return f'${p:,.2f}' if p >= 1 else f'${p:.6f}'

def fmt_mc(m):
    m = m or 0
    if m >= 1e12: return f'${m/1e12:.2f}T'
    if m >= 1e9:  return f'${m/1e9:.2f}B'
    return f'${m/1e6:.2f}M'

fetch_time = coins[0]['fetched_at']
avg_color  = 'green' if avg_change >= 0 else 'red'

# Build table rows
rows_html = ''
for c in coins:
    chg   = c['price_change_24h'] or 0
    cls   = 'positive' if chg >= 0 else 'negative'
    arrow = '▲' if chg >= 0 else '▼'
    rows_html += (
        "<tr>"
        f"<td class='rank'>#{c['rank']}</td>"
        f"<td class='coin-cell'><img src='{c['image']}' onerror=\"this.style.display='none'\"/>"
        f"<div><span class='coin-name'>{c['name']}</span><br/><span class='coin-sym'>{c['symbol']}</span></div></td>"
        f"<td class='price'>{fmt_price(c['current_price_usd'])}</td>"
        f"<td class='change {cls}'>{arrow} {abs(chg):.2f}%</td>"
        f"<td>{fmt_mc(c['market_cap_usd'])}</td>"
        f"<td>{fmt_mc(c['total_volume_usd'])}</td>"
        f"<td>{fmt_price(c['high_24h_usd'])}</td>"
        f"<td>{fmt_price(c['low_24h_usd'])}</td>"
        "</tr>"
    )

# Build gainers cards
gainers_html = ''
for c in gainers[:5]:
    gainers_html += (
        f"<div class='mover-card gain'>"
        f"<img src='{c['image']}' onerror=\"this.style.display='none'\"/>"
        f"<div class='mover-info'><span class='mover-name'>{c['name']}</span>"
        f"<span class='mover-sym'>{c['symbol']}</span></div>"
        f"<span class='mover-chg gain-text'>+{c['price_change_24h']:.2f}%</span></div>"
    )

# Build losers cards
losers_html = ''
for c in losers[:5]:
    losers_html += (
        f"<div class='mover-card loss'>"
        f"<img src='{c['image']}' onerror=\"this.style.display='none'\"/>"
        f"<div class='mover-info'><span class='mover-name'>{c['name']}</span>"
        f"<span class='mover-sym'>{c['symbol']}</span></div>"
        f"<span class='mover-chg loss-text'>{c['price_change_24h']:.2f}%</span></div>"
    )

print('✅ Building HTML dashboard...')

✅ Building HTML dashboard...


In [ ]:
css = """
:root{--bg:#0a0a0f;--card:#111118;--border:#1e1e2e;--accent:#f7931a;--accent2:#00d4aa;--green:#00e676;--red:#ff5252;--text:#e8e8f0;--muted:#6b6b80;}
*{margin:0;padding:0;box-sizing:border-box;}
body{background:var(--bg);color:var(--text);font-family:'Space Mono',monospace;min-height:100vh;}
.header{background:linear-gradient(135deg,#0a0a0f,#12121f,#0a0a0f);border-bottom:1px solid var(--border);padding:32px 48px;display:flex;align-items:center;justify-content:space-between;position:relative;overflow:hidden;}
.header::before{content:'';position:absolute;top:-60px;left:-60px;width:300px;height:300px;background:radial-gradient(circle,rgba(247,147,26,0.08),transparent 70%);}
.header-left h1{font-family:'Syne',sans-serif;font-size:2.2rem;font-weight:800;color:var(--accent);letter-spacing:-1px;}
.header-left p{color:var(--muted);font-size:0.75rem;margin-top:4px;}
.header-right{text-align:right;font-size:0.72rem;color:var(--muted);line-height:1.8;}
.live-badge{display:inline-flex;align-items:center;gap:6px;background:rgba(0,214,170,0.1);border:1px solid rgba(0,214,170,0.3);color:var(--accent2);padding:4px 12px;border-radius:20px;font-size:0.7rem;margin-bottom:6px;}
.live-dot{width:6px;height:6px;background:var(--accent2);border-radius:50%;animation:pulse 1.5s infinite;}
@keyframes pulse{0%,100%{opacity:1;transform:scale(1);}50%{opacity:0.4;transform:scale(1.4);}}
.main{padding:32px 48px;max-width:1400px;margin:0 auto;}
.stats-grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(180px,1fr));gap:16px;margin-bottom:32px;}
.stat-card{background:var(--card);border:1px solid var(--border);border-radius:12px;padding:20px;transition:border-color 0.2s;}
.stat-card:hover{border-color:var(--accent);}
.stat-label{font-size:0.65rem;color:var(--muted);text-transform:uppercase;letter-spacing:1.5px;margin-bottom:8px;}
.stat-value{font-family:'Syne',sans-serif;font-size:1.4rem;font-weight:800;color:var(--text);}
.stat-value.accent{color:var(--accent);}.stat-value.green{color:var(--green);}.stat-value.red{color:var(--red);}.stat-value.teal{color:var(--accent2);}
.movers-grid{display:grid;grid-template-columns:1fr 1fr;gap:24px;margin-bottom:32px;}
.movers-box{background:var(--card);border:1px solid var(--border);border-radius:12px;padding:24px;}
.movers-title{font-family:'Syne',sans-serif;font-size:0.85rem;font-weight:600;text-transform:uppercase;letter-spacing:2px;margin-bottom:16px;}
.movers-title.green{color:var(--green);}.movers-title.red{color:var(--red);}
.mover-card{display:flex;align-items:center;gap:12px;padding:10px 14px;border-radius:8px;margin-bottom:8px;}
.mover-card.gain{background:rgba(0,230,118,0.06);border:1px solid rgba(0,230,118,0.15);}
.mover-card.loss{background:rgba(255,82,82,0.06);border:1px solid rgba(255,82,82,0.15);}
.mover-card img{width:28px;height:28px;border-radius:50%;}
.mover-info{flex:1;}.mover-name{display:block;font-size:0.8rem;font-weight:700;}.mover-sym{font-size:0.65rem;color:var(--muted);}
.mover-chg{font-family:'Syne',sans-serif;font-weight:700;font-size:0.9rem;}.gain-text{color:var(--green);}.loss-text{color:var(--red);}
.table-wrap{background:var(--card);border:1px solid var(--border);border-radius:12px;overflow:hidden;}
.table-header{display:flex;align-items:center;justify-content:space-between;padding:20px 24px;border-bottom:1px solid var(--border);}
.table-title{font-family:'Syne',sans-serif;font-weight:700;font-size:1rem;}
.search-box{background:var(--bg);border:1px solid var(--border);border-radius:8px;padding:8px 14px;color:var(--text);font-family:'Space Mono',monospace;font-size:0.75rem;outline:none;width:220px;transition:border-color 0.2s;}
.search-box:focus{border-color:var(--accent);}.search-box::placeholder{color:var(--muted);}
table{width:100%;border-collapse:collapse;font-size:0.78rem;}
thead th{background:rgba(255,255,255,0.02);padding:12px 16px;text-align:left;font-size:0.65rem;text-transform:uppercase;letter-spacing:1px;color:var(--muted);border-bottom:1px solid var(--border);}
tbody tr{border-bottom:1px solid rgba(255,255,255,0.03);transition:background 0.15s;}
tbody tr:hover{background:rgba(247,147,26,0.04);}
td{padding:13px 16px;vertical-align:middle;}
.rank{color:var(--muted);font-size:0.7rem;}
.coin-cell{display:flex;align-items:center;gap:10px;}.coin-cell img{width:24px;height:24px;border-radius:50%;}
.coin-name{font-weight:700;font-size:0.82rem;}.coin-sym{font-size:0.65rem;color:var(--muted);}
.price{font-weight:700;font-family:'Syne',sans-serif;}.change{font-weight:700;}.positive{color:var(--green);}.negative{color:var(--red);}
.footer{text-align:center;padding:32px;color:var(--muted);font-size:0.7rem;border-top:1px solid var(--border);margin-top:40px;}
.footer span{color:var(--accent);}
.bar-bg{background:rgba(255,82,82,0.3);border-radius:4px;height:6px;margin-top:10px;overflow:hidden;}
.bar-fill{height:100%;background:linear-gradient(90deg,var(--red),var(--green));border-radius:4px;}
"""

html = f"""<!DOCTYPE html>
<html lang='en'>
<head>
<meta charset='UTF-8'/>
<meta name='viewport' content='width=device-width,initial-scale=1.0'/>
<title>Crypto Price Snapshot</title>
<link href='https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=Syne:wght@400;600;800&display=swap' rel='stylesheet'/>
<style>{css}</style>
</head>
<body>
<div class='header'>
  <div class='header-left'>
    <h1>🪙 Crypto Price Snapshot</h1>
    <p>Lalit Gupta Batch T-361 &nbsp;|&nbsp; CoinGecko API &nbsp;|&nbsp; Top 50 Coins</p>
  </div>
  <div class='header-right'>
    <div class='live-badge'><span class='live-dot'></span> LIVE DATA</div><br/>
    Fetched: {fetch_time}
  </div>
</div>
<div class='main'>
  <div class='stats-grid'>
    <div class='stat-card'><div class='stat-label'>Total Coins</div><div class='stat-value accent'>50</div></div>
    <div class='stat-card'><div class='stat-label'>Avg Price (USD)</div><div class='stat-value'>${avg_price:,.2f}</div></div>
    <div class='stat-card'><div class='stat-label'>Avg 24h Change</div><div class='stat-value {avg_color}'>{avg_change:+.2f}%</div></div>
    <div class='stat-card'><div class='stat-label'>Coins Up 🟢</div><div class='stat-value green'>{gainers_count}</div></div>
    <div class='stat-card'><div class='stat-label'>Coins Down 🔴</div><div class='stat-value red'>{losers_count}</div></div>
    <div class='stat-card'><div class='stat-label'>Market Mood</div><div class='stat-value teal' style='font-size:1rem'>{mood}</div><div class='bar-bg'><div class='bar-fill' style='width:{green_pct:.0f}%'></div></div></div>
    <div class='stat-card'><div class='stat-label'>Top Gainer</div><div class='stat-value green' style='font-size:1rem'>{gainers[0]['symbol']} +{gainers[0]['price_change_24h']:.1f}%</div></div>
    <div class='stat-card'><div class='stat-label'>Top Loser</div><div class='stat-value red' style='font-size:1rem'>{losers[0]['symbol']} {losers[0]['price_change_24h']:.1f}%</div></div>
  </div>
  <div class='movers-grid'>
    <div class='movers-box'><div class='movers-title green'>▲ Top 5 Gainers (24h)</div>{gainers_html}</div>
    <div class='movers-box'><div class='movers-title red'>▼ Top 5 Losers (24h)</div>{losers_html}</div>
  </div>
  <div class='table-wrap'>
    <div class='table-header'>
      <span class='table-title'>All 50 Cryptocurrencies</span>
      <input class='search-box' type='text' id='searchInput' placeholder='🔍 Search coin...' onkeyup='filterTable()'/>
    </div>
    <table id='cryptoTable'>
      <thead><tr><th>#</th><th>Coin</th><th>Price (USD)</th><th>24h Change</th><th>Market Cap</th><th>Volume (24h)</th><th>High 24h</th><th>Low 24h</th></tr></thead>
      <tbody id='tableBody'>{rows_html}</tbody>
    </table>
  </div>
</div>
<div class='footer'>Built with <span>Python</span> | Data from <span>CoinGecko API</span> | TY Data Science Project | {fetch_time}</div>
<script>
function filterTable(){{var q=document.getElementById('searchInput').value.toLowerCase();document.querySelectorAll('#tableBody tr').forEach(function(r){{r.style.display=r.textContent.toLowerCase().includes(q)?'':'none';}});}}
</script>
</body></html>"""

html_file = os.path.abspath('crypto_dashboard.html')
with open(html_file, 'w', encoding='utf-8') as f:
    f.write(html)

print('✅ Dashboard saved!')
print(f'📂 Location: {html_file}')
print('🌐 Opening browser now...')
webbrowser.open('file:///' + html_file.replace('\\', '/'))

✅ Dashboard saved!
📂 Location: c:\Users\Gaurav Gupta\OneDrive\Desktop\crypto-project\crypto_dashboard.html
🌐 Opening browser now...


True

## ✅ Project Complete!

Your browser dashboard should have opened automatically!

### Files Generated:
| File | Description |
|------|-------------|
| `crypto_data.csv` | Raw data of all 50 coins |
| `crypto_dashboard.html` | 🌐 Browser dashboard — double-click to open |

### Key Concepts Used:
- REST API with `requests`
- JSON parsing
- Error handling with `try/except` and `safe()` helper
- CSV read/write
- HTML dashboard generation
- Data analysis without pandas